# Fine-tuning Whisper для русского ASR


Современная версия notebook для дообучения `openai/whisper-small` на русском аудиодатасете.

Обновлено под стек августа 2026:

- `transformers 5.15.x`
- `datasets 5.0.x` и TorchCodec
- `accelerate 1.14.x`
- `jiwer 4.x`
- Python 3.10+

### Что изменено по сравнению со старой версией

- `Audio` теперь обрабатывается как `AudioDecoder`, а не как словарь `{"array", "sampling_rate"}`.
- Не создаётся огромный промежуточный датасет с заранее рассчитанными 30-секундными log-Mel признаками.
- Декодирование, ресемплинг и feature extraction выполняются динамически в data collator.
- Используются `AutoProcessor` и `AutoModelForSpeechSeq2Seq`.
- `Seq2SeqTrainer` получает `processing_class`, а не устаревший аргумент `tokenizer`.
- Используется актуальный `eval_strategy`.
- На совместимых NVIDIA GPU автоматически используется BF16 и TF32.
- Push в Hugging Face Hub отключён по умолчанию.


## 1. Зависимости


PyTorch лучше устанавливать в составе CUDA-образа/окружения, подходящего вашей GPU. Эта ячейка обновляет только библиотеки Hugging Face и ASR-зависимости.

> После обновления пакетов перезапустите kernel, если они уже были импортированы в текущей сессии.


In [1]:
%pip install -U \
    "transformers==5.15.0" \
    "datasets[audio]==5.0.1" \
    "accelerate==1.14.0" \
    "jiwer==4.0.0" \
    "huggingface_hub"


Note: you may need to restart the kernel to use updated packages.


## 2. Импорты и проверка среды


In [2]:
import os
import sys
from importlib.metadata import version as package_version
from dataclasses import dataclass
from typing import Any

import accelerate
import datasets
import jiwer
import torch
import transformers
from datasets import Audio, DatasetDict, concatenate_datasets, load_dataset
from packaging.version import Version
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

try:
    import torchcodec
    torchcodec_version = torchcodec.__version__
except Exception:
    torchcodec_version = "not available"

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"Accelerate:   {accelerate.__version__}")
print(f"JiWER:        {package_version('jiwer')}")
print(f"TorchCodec:   {torchcodec_version}")

assert Version(transformers.__version__) >= Version("5.15.0")
assert Version(datasets.__version__) >= Version("5.0.1")
assert Version(accelerate.__version__) >= Version("1.14.0")

if torch.cuda.is_available():
    print(f"\nGPU:          {torch.cuda.get_device_name(0)}")
    print(f"CUDA:         {torch.version.cuda}")
    print(f"BF16:         {torch.cuda.is_bf16_supported()}")
else:
    print("\nCUDA GPU is not available.")


Python:       3.10.12
PyTorch:      2.13.0+cu130
Transformers: 5.15.0
Datasets:     5.0.1
Accelerate:   1.14.0
JiWER:        4.0.0
TorchCodec:   0.15.0+cpu

GPU:          NVIDIA GeForce RTX 5090
CUDA:         13.0
BF16:         True


## 3. Конфигурация и локальные outputs


Скачиваемые модели, tokenizer'ы и датасеты используют стандартный общий Hugging Face cache:

```text
~/.cache/huggingface/
```

Notebook не переопределяет `HF_HOME`, `HF_HUB_CACHE`, `HF_DATASETS_CACHE` и не передаёт `cache_dir` в `load_dataset()`.

Только результаты конкретного обучения сохраняются рядом с notebook:

```text
asr/
├── Whisper_fine_tune_2026.ipynb
└── outputs/
    └── whisper-small-ru/
        ├── checkpoint-*/
        ├── config.json
        ├── model.safetensors
        └── ...
```

Для Git достаточно универсального правила:

```gitignore
**/outputs/
```

Для короткой проверки notebook установите, например:

```python
MAX_TRAIN_SAMPLES = 10_000
MAX_EVAL_SAMPLES = 1_000
MAX_STEPS = 100
```

Для полноценного обучения оставьте `MAX_*_SAMPLES = None`.


In [3]:
from pathlib import Path

SEED = 42

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
# Hugging Face models and datasets use the library's default shared cache
# (~/.cache/huggingface). Only training outputs are stored next to the notebook.
cwd = Path.cwd().resolve()

if cwd.name == "asr":
    NOTEBOOK_DIR = cwd
elif (cwd / "notebooks" / "finetuning" / "asr").is_dir():
    NOTEBOOK_DIR = (cwd / "notebooks" / "finetuning" / "asr").resolve()
else:
    raise RuntimeError(
        "Cannot locate notebooks/finetuning/asr. "
        f"Current working directory: {cwd}"
    )

OUTPUTS_DIR = NOTEBOOK_DIR / "outputs"
MODEL_OUTPUT_DIR = OUTPUTS_DIR / "whisper-small-ru"

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Model output:        {MODEL_OUTPUT_DIR}")

# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------
DATASET_ID = "artyomboyko/common_voice_21_0_ru"
DATASET_CONFIG = "default"
TRAIN_SPLITS = ("train", "validation")
EVAL_SPLIT = "test"
AUDIO_COLUMN = "audio"
TEXT_COLUMN = "sentence"
DURATION_COLUMN = "duration[ms]"

# -----------------------------------------------------------------------------
# Model
# -----------------------------------------------------------------------------
MODEL_ID = "openai/whisper-small"
LANGUAGE = "ru"
TASK = "transcribe"

OUTPUT_DIR = str(MODEL_OUTPUT_DIR)
PUSH_TO_HUB = False
HUB_MODEL_ID = "artyomboyko/whisper-small-ru-v5"

TARGET_SAMPLE_RATE = 16_000
MIN_DURATION_SECONDS = 0.1
MAX_DURATION_SECONDS = 30.0

# None = use the entire split.
MAX_TRAIN_SAMPLES = 5_000
MAX_EVAL_SAMPLES = 1_500

# Training.
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-6
WARMUP_STEPS = 500
MAX_STEPS = 2_500

# Preprocessing.
# GPU feature extraction uses batched torch.stft and is especially useful
# on modern NVIDIA GPUs. DataLoader workers must be disabled in this mode.
FEATURE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CPU_DATALOADER_WORKERS = min(4, os.cpu_count() or 1)

set_seed(SEED)

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")


Notebook directory: /workspace/notebooks/finetuning/asr
Model output:        /workspace/notebooks/finetuning/asr/outputs/whisper-small-ru


### Hugging Face authentication


Для чтения публичного датасета вход не требуется. Авторизуйтесь только если датасет/модель gated или если включаете `PUSH_TO_HUB`.

Предпочтительный вариант для локальной среды:

```bash
hf auth login
```

Токен не хранится в notebook.


## 4. Загрузка датасета


`train` и `validation` объединяются в обучающий split.

`load_dataset()` использует стандартный Hugging Face cache. Это позволяет всем notebook'ам в контейнере переиспользовать уже скачанные датасеты и не создавать отдельные копии рядом с каждым экспериментом.

Метаданные, которые не нужны модели, будут удалены после фильтрации длительности.


In [4]:
raw = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
)

available_train_splits = [name for name in TRAIN_SPLITS if name in raw]
if not available_train_splits:
    raise ValueError(f"None of TRAIN_SPLITS={TRAIN_SPLITS} found. Available: {list(raw.keys())}")
if EVAL_SPLIT not in raw:
    raise ValueError(f"EVAL_SPLIT={EVAL_SPLIT!r} not found. Available: {list(raw.keys())}")

train_parts = [raw[name] for name in available_train_splits]
train_dataset = train_parts[0] if len(train_parts) == 1 else concatenate_datasets(train_parts)

common_voice = DatasetDict(
    train=train_dataset,
    eval=raw[EVAL_SPLIT],
)

print(common_voice)
print("Columns:", common_voice["train"].column_names)


DatasetDict({
    train: Dataset({
        features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]'],
        num_rows: 36793
    })
    eval: Dataset({
        features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]'],
        num_rows: 10229
    })
})
Columns: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]']


### Опциональное ограничение объёма данных


При заданном лимите выборка сначала перемешивается с фиксированным seed. Это удобно для smoke test без изменения остального notebook.


In [5]:
def limit_dataset(dataset, max_samples):
    if max_samples is None:
        return dataset
    count = min(max_samples, len(dataset))
    return dataset.shuffle(seed=SEED).select(range(count))

common_voice["train"] = limit_dataset(common_voice["train"], MAX_TRAIN_SAMPLES)
common_voice["eval"] = limit_dataset(common_voice["eval"], MAX_EVAL_SAMPLES)

print(common_voice)


DatasetDict({
    train: Dataset({
        features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]'],
        num_rows: 5000
    })
    eval: Dataset({
        features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]'],
        num_rows: 1500
    })
})


### Фильтрация слишком длинных записей


Whisper работает с окнами до 30 секунд. Если в датасете есть готовая колонка длительности, фильтруем по ней без декодирования аудио.


In [6]:
if DURATION_COLUMN in common_voice["train"].column_names:
    min_ms = int(MIN_DURATION_SECONDS * 1000)
    max_ms = int(MAX_DURATION_SECONDS * 1000)

    def duration_in_range(duration_ms):
        return duration_ms is not None and min_ms <= duration_ms <= max_ms

    filter_num_proc = min(4, os.cpu_count() or 1)

    common_voice = common_voice.filter(
        duration_in_range,
        input_columns=[DURATION_COLUMN],
        num_proc=filter_num_proc,
        desc="Filtering audio by duration",
    )

print(common_voice)


Filtering audio by duration (num_proc=4):   0%|          | 0/5000 [00:00<?, ? examples/s]

Filtering audio by duration (num_proc=4):   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]'],
        num_rows: 5000
    })
    eval: Dataset({
        features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'locale', 'segment', 'variant', 'duration[ms]'],
        num_rows: 1500
    })
})


### Оставляем только поля, необходимые для обучения


Это уменьшает объём данных, передаваемых между `Dataset` и data collator.


In [7]:
required_columns = {AUDIO_COLUMN, TEXT_COLUMN}

for split in common_voice:
    missing = required_columns.difference(common_voice[split].column_names)
    if missing:
        raise ValueError(f"{split}: missing required columns: {sorted(missing)}")

    columns_to_remove = [
        column
        for column in common_voice[split].column_names
        if column not in required_columns
    ]
    common_voice[split] = common_voice[split].remove_columns(columns_to_remove)

print(common_voice)


DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 5000
    })
    eval: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 1500
    })
})


## 5. Processor и модель


`AutoProcessor` объединяет tokenizer и feature extractor. Для многоязычного Whisper явно задаются язык и задача генерации.


In [8]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID)

processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task=TASK)

if getattr(model.generation_config, "is_multilingual", False):
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK

# Актуальная схема Transformers: язык/задача задаются через generation_config,
# а forced_decoder_ids отключаются.
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

print(f"Model: {MODEL_ID}")
print(f"Feature extractor sampling rate: {processor.feature_extractor.sampling_rate} Hz")


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model: openai/whisper-small
Feature extractor sampling rate: 16000 Hz


## 6. Ресемплинг через Datasets + TorchCodec


Начиная с новых версий `datasets`, доступ к `audio` возвращает `AudioDecoder`. `cast_column()` не переписывает весь датасет: ресемплинг до 16 кГц выполняется при декодировании.


In [9]:
for split in common_voice:
    common_voice[split] = common_voice[split].cast_column(
        AUDIO_COLUMN,
        Audio(sampling_rate=TARGET_SAMPLE_RATE),
    )

audio_decoder = common_voice["train"][0][AUDIO_COLUMN]
samples = audio_decoder.get_all_samples()

print("Decoder type:", type(audio_decoder).__name__)
print("Sample rate:", samples.sample_rate)
print("Tensor shape:", tuple(samples.data.shape))
print("Text:", common_voice["train"][0][TEXT_COLUMN])


Decoder type: AudioDecoder
Sample rate: 16000
Tensor shape: (1, 82944)
Text: Она девка, а ты барин, — проговорил он, подергиваясь шеей.


## 7. Data collator с динамическим feature extraction


В старой версии notebook log-Mel признаки вычислялись для всего датасета через `Dataset.map()` и сохранялись на диск. Для Whisper это дорого: feature extractor по умолчанию формирует признаки 30-секундного окна для каждого примера.

Здесь признаки рассчитываются **только для текущего batch**:

1. TorchCodec декодирует и ресемплирует аудио.
2. Каналы при необходимости сводятся в mono.
3. `WhisperFeatureExtractor` вычисляет log-Mel spectrograms батчем.
4. Тексты токенизируются батчем.
5. Padding labels заменяется на `-100`.

Если доступна CUDA, log-Mel вычисляется через GPU-ускоренный `torch.stft`.


In [10]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    sampling_rate: int = 16_000
    feature_device: str = "cpu"

    @staticmethod
    def _decode_mono(audio_decoder):
        samples = audio_decoder.get_all_samples()
        waveform = samples.data

        # TorchCodec: (channels, samples). Whisper ожидает mono waveform.
        if waveform.ndim == 2:
            if waveform.shape[0] == 1:
                waveform = waveform.squeeze(0)
            else:
                waveform = waveform.mean(dim=0)

        return waveform.detach().cpu().float().numpy(), int(samples.sample_rate)

    def __call__(self, features):
        waveforms = []
        texts = []

        for feature in features:
            waveform, sample_rate = self._decode_mono(feature[AUDIO_COLUMN])

            if sample_rate != self.sampling_rate:
                raise ValueError(
                    f"Expected {self.sampling_rate} Hz after cast_column(), got {sample_rate} Hz."
                )

            waveforms.append(waveform)
            texts.append(feature[TEXT_COLUMN])

        # В Transformers 5.x WhisperFeatureExtractor умеет батчевый PyTorch STFT
        # и может выполнять его на CUDA.
        batch = self.processor.feature_extractor(
            waveforms,
            sampling_rate=self.sampling_rate,
            return_attention_mask=True,
            return_tensors="pt",
            device=self.feature_device,
        )

        labels_batch = self.processor.tokenizer(
            texts,
            padding=True,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        # decoder_start_token_id Trainer/model добавит самостоятельно.
        if labels.shape[1] > 0 and (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
    sampling_rate=TARGET_SAMPLE_RATE,
    feature_device=FEATURE_DEVICE,
)

print("Feature extraction device:", FEATURE_DEVICE)


Feature extraction device: cuda


### Проверка collator до запуска обучения


Эта ячейка ловит большинство ошибок аудио API, TorchCodec, tokenizer и shapes до дорогостоящего запуска `Trainer`.


In [11]:
test_batch = data_collator(
    [common_voice["train"][0], common_voice["train"][1]]
)

for key, value in test_batch.items():
    print(f"{key:16s}: {tuple(value.shape)} {value.dtype}")


input_features  : (2, 80, 3000) torch.float32
attention_mask  : (2, 3000) torch.int32
labels          : (2, 31) torch.int64


## 8. WER


Для одной метрики нет необходимости тянуть отдельную библиотеку `evaluate`: WER вычисляется напрямую через `jiwer`.


In [12]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    predictions = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True,
    )
    references = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True,
    )

    return {
        "wer": 100.0 * jiwer.wer(references, predictions),
    }


## 9. TrainingArguments


Оптимизации по умолчанию:

- BF16 на совместимых NVIDIA GPU; FP16 используется как fallback.
- TF32 для быстрых матричных операций на современных NVIDIA GPU.
- fused AdamW на CUDA.
- `num_beams=1` во время промежуточной WER-оценки.
- ограниченное количество checkpoints.
- при GPU feature extraction `dataloader_num_workers=0`, чтобы CUDA не использовалась из forked worker processes.

Если памяти GPU недостаточно, уменьшите `TRAIN_BATCH_SIZE` и увеличьте `GRADIENT_ACCUMULATION_STEPS`.


In [13]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# CUDA нельзя безопасно использовать из forked DataLoader workers.
dataloader_workers = 0 if FEATURE_DEVICE == "cuda" else CPU_DATALOADER_WORKERS

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,

    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_strategy="steps",
    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=1,
    eval_accumulation_steps=4,

    bf16=use_bf16,
    fp16=use_fp16,
    tf32=True if torch.cuda.is_available() else None,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,

    # Нужны raw columns audio/sentence для нашего динамического collator.
    remove_unused_columns=False,

    dataloader_num_workers=dataloader_workers,
    dataloader_pin_memory=torch.cuda.is_available(),
    dataloader_persistent_workers=dataloader_workers > 0,
    dataloader_prefetch_factor=2 if dataloader_workers > 0 else None,

    report_to="none",

    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
)

print("BF16:", use_bf16)
print("FP16:", use_fp16)
print("DataLoader workers:", dataloader_workers)


BF16: True
FP16: False
DataLoader workers: 0


## 10. Trainer


In [14]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["eval"],
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 11. Обучение


Для продолжения с последнего checkpoint используйте:

```python
trainer.train(resume_from_checkpoint=True)
```


In [15]:
train_result = trainer.train()

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()


Step,Training Loss,Validation Loss,Wer
500,0.212712,0.263109,20.359522
1000,0.133768,0.241671,19.316267
1500,0.127678,0.238010,18.979215
2000,0.073860,0.237820,19.075516
2500,0.086093,0.238159,19.099591


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


***** train metrics *****
  epoch                    =        7.9872
  total_flos               = 10735593019GF
  train_loss               =        0.1856
  train_runtime            =    0:20:30.86
  train_samples_per_second =        32.497
  train_steps_per_second   =         2.031


## 12. Финальная оценка и сохранение


`OUTPUT_DIR` указывает на:

```text
outputs/whisper-small-ru/
```

В этой директории сохраняются checkpoints, финальные веса модели, конфигурация и processor. Скачанная исходная модель и датасет остаются в стандартном Hugging Face cache.


In [16]:
eval_metrics = trainer.evaluate()

trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

eval_metrics


Training Loss,Validation Loss,Step,Wer
0.086093,0.238010,2500,18.979215


***** eval metrics *****
  eval_loss =   0.238
  eval_wer  = 18.9792


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.23801016807556152, 'eval_wer': 18.97921515127197}

## 13. Model card / Hugging Face Hub


При `PUSH_TO_HUB=False` создаётся локальный model card. Чтобы опубликовать модель, авторизуйтесь через `hf auth login`, установите `PUSH_TO_HUB=True` и перезапустите ячейку с `TrainingArguments` до создания `Trainer`.


In [17]:
model_card_kwargs = {
    "finetuned_from": MODEL_ID,
    "tasks": "automatic-speech-recognition",
    "dataset_tags": DATASET_ID,
    "dataset": DATASET_ID,
    "language": LANGUAGE,
}

if PUSH_TO_HUB:
    trainer.push_to_hub(**model_card_kwargs)
else:
    trainer.create_model_card(**model_card_kwargs)
    print(f"Model and processor saved locally to: {OUTPUT_DIR}")


Model and processor saved locally to: /workspace/notebooks/finetuning/asr/outputs/whisper-small-ru


## 14. Smoke test inference


In [18]:
example = common_voice["eval"][0]
audio_samples = example[AUDIO_COLUMN].get_all_samples()

waveform = audio_samples.data
if waveform.ndim == 2:
    waveform = waveform.squeeze(0) if waveform.shape[0] == 1 else waveform.mean(dim=0)

inputs = processor(
    waveform.cpu().float().numpy(),
    sampling_rate=int(audio_samples.sample_rate),
    return_tensors="pt",
)

device = trainer.model.device
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.inference_mode():
    generated_ids = trainer.model.generate(
        **inputs,
        language=LANGUAGE,
        task=TASK,
        max_new_tokens=225,
    )

prediction = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)[0]

print("Reference :", example[TEXT_COLUMN])
print("Prediction:", prediction)


[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reference : Облонский, видимо, страдал.
Prediction: Аблонский, видимо, страдал.
